# Lesson 3 — Placing Loads on the Beam

**Course:** Python for Structural Engineers — Simply Supported Beam  
**Prerequisites:** Lessons 1 and 2  
**You will learn:**
- Use Python **dictionaries** to group related data (load type, position, magnitude)
- Write your own **functions** with `def`
- Use **buttons** in a widget to add and remove loads interactively
- Draw point-load arrows and UDL arrow-banks on the beam diagram

**Time estimate:** 60 minutes

---

## Sign Convention — Agree Before You Calculate

In structural analysis, we must all agree on signs before writing a single equation. Throughout this course:

| Quantity | Positive direction |
|----------|--------------------|
| Load     | **Upward** (+) |
| Load (gravity/downward) | **Negative** (−) |
| Reaction | **Upward** (+) |
| Bending moment (sagging) | **Positive** (+) |
| Bending moment (hogging) | **Negative** (−) |

So a **10 kN downward point load** will always be written as `magnitude = -10.0`.

---

## 3.1 Dictionaries — Grouping Related Data

A **dictionary** stores pairs of `key: value`. It's perfect for describing a load, because each load has several properties that belong together:

In [ ]:
# A single point load described as a dictionary
load1 = {
    "type"      : "point",   # what kind of load
    "x"         : 5.0,       # position along beam (m)
    "magnitude" : -20.0      # kN  (negative = downward)
}

# Access individual values with the key inside square brackets
print(f"Load type : {load1['type']}")
print(f"Position  : {load1['x']} m")
print(f"Magnitude : {load1['magnitude']} kN")

In [ ]:
# A UDL has a start position, end position, and intensity
load2 = {
    "type" : "udl",
    "x1"   : 2.0,    # start of UDL (m)
    "x2"   : 8.0,    # end of UDL (m)
    "w"    : -5.0    # kN/m  (negative = downward)
}

# Total force from the UDL:
total_udl_force = load2["w"] * (load2["x2"] - load2["x1"])
print(f"Total UDL force = {total_udl_force} kN")

We store all loads in a **list of dictionaries**:

In [ ]:
loads = [load1, load2]

# Loop over all loads and print a summary
for i, load in enumerate(loads):
    if load["type"] == "point":
        print(f"Load {i+1}: Point load  {load['magnitude']:+.1f} kN  at x = {load['x']:.1f} m")
    elif load["type"] == "udl":
        print(f"Load {i+1}: UDL  {load['w']:+.1f} kN/m  from x = {load['x1']:.1f} m to x = {load['x2']:.1f} m")

---

## 3.2 Writing Functions with `def`

A **function** is a named block of code you can call whenever you need it. This avoids copying the same code many times.

Syntax:
```python
def function_name(parameter1, parameter2):
    # code goes here (indented)
    return result
```

Let's write a function to validate and add a point load:

In [ ]:
def add_point_load(loads_list, x, magnitude, L_total):
    """Add a point load to the load list after validating its position."""
    if x < 0 or x > L_total:
        print(f"Error: position {x} m is outside the beam (0 to {L_total} m).")
        return
    if magnitude == 0:
        print("Warning: zero magnitude load ignored.")
        return
    loads_list.append({"type": "point", "x": x, "magnitude": magnitude})
    print(f"Added point load {magnitude:+.1f} kN at x = {x:.1f} m")


def add_udl(loads_list, x1, x2, w, L_total):
    """Add a UDL to the load list after validating its extent."""
    x1 = max(0.0, x1)
    x2 = min(L_total, x2)
    if x2 <= x1:
        print("Error: UDL end must be to the right of UDL start.")
        return
    loads_list.append({"type": "udl", "x1": x1, "x2": x2, "w": w})
    print(f"Added UDL {w:+.1f} kN/m from x = {x1:.1f} m to x = {x2:.1f} m")


# Test the functions
my_loads = []
add_point_load(my_loads, 5.0, -20.0, L_total=10.0)
add_udl(my_loads, 2.0, 8.0, -5.0, L_total=10.0)
add_point_load(my_loads, 15.0, -10.0, L_total=10.0)   # should show error

print(f"\nTotal loads defined: {len(my_loads)}")

---

## 3.3 Drawing Loads on the Beam

Now let's draw the beam from Lesson 2 with load arrows overlaid.

A key matplotlib trick: **`ax.annotate()`** draws arrows between two points.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import sys
sys.path.insert(0, "../shared")
from plot_helpers import draw_beam, draw_loads

geometry = {"L_total": 10.0, "x_A": 2.0, "x_B": 8.0}
loads_example = [
    {"type": "point", "x": 5.0, "magnitude": -20.0},
    {"type": "udl",   "x1": 2.0, "x2": 8.0, "w": -5.0},
    {"type": "point", "x": 1.0, "magnitude": -8.0},    # on left cantilever
]

fig, ax = plt.subplots(figsize=(13, 4))
draw_beam(ax, geometry)
draw_loads(ax, loads_example, geometry["L_total"])
ax.set_title("Beam with applied loads", fontsize=11, pad=8)
plt.tight_layout()
plt.show()

---

## 3.4 Interactive Load Builder Widget

Now let's build a widget where you can **add** loads one at a time using a button, and **clear** them all.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import sys
sys.path.insert(0, "../shared")
from plot_helpers import draw_beam, draw_loads

%matplotlib inline

# ── Beam geometry (fixed for now; becomes interactive in L04) ─────────────
geometry = {"L_total": 10.0, "x_A": 2.0, "x_B": 8.0}

# ── Shared state: the load list (mutable, shared between callbacks) ───────
live_loads = []

# ── Output area ───────────────────────────────────────────────────────────
out_plot   = widgets.Output()
out_status = widgets.Output()

# ── Controls ──────────────────────────────────────────────────────────────
style  = {"description_width": "160px"}
layout = widgets.Layout(width="440px")

dd_type = widgets.Dropdown(
    options=[("Point load", "point"), ("UDL", "udl")],
    value="point", description="Load type:", style=style, layout=layout
)

# Point load controls
sl_x   = widgets.FloatSlider(value=5.0, min=0.0, max=10.0, step=0.5,
                              description="Position x (m):", style=style, layout=layout)
sl_P   = widgets.FloatSlider(value=-20.0, min=-100.0, max=100.0, step=5.0,
                              description="Magnitude (kN):", style=style, layout=layout)

# UDL-specific controls (hidden until UDL is chosen)
sl_x1  = widgets.FloatSlider(value=2.0, min=0.0, max=9.5, step=0.5,
                              description="UDL start x1 (m):", style=style, layout=layout)
sl_x2  = widgets.FloatSlider(value=8.0, min=0.5, max=10.0, step=0.5,
                              description="UDL end x2 (m):", style=style, layout=layout)
sl_w   = widgets.FloatSlider(value=-5.0, min=-40.0, max=40.0, step=1.0,
                              description="Intensity (kN/m):", style=style, layout=layout)

# Buttons
btn_add   = widgets.Button(description="Add Load",    button_style="primary")
btn_clear = widgets.Button(description="Clear All",   button_style="danger")

# Show/hide UDL controls based on dropdown
udl_controls = widgets.VBox([sl_x1, sl_x2, sl_w])
pt_controls  = widgets.VBox([sl_x, sl_P])

def on_type_change(change):
    if change["new"] == "point":
        pt_controls.layout.display  = ""
        udl_controls.layout.display = "none"
    else:
        pt_controls.layout.display  = "none"
        udl_controls.layout.display = ""

dd_type.observe(on_type_change, names="value")
udl_controls.layout.display = "none"   # hidden initially

# ── Drawing function ──────────────────────────────────────────────────────
def redraw():
    with out_plot:
        out_plot.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(13, 4.2))
        draw_beam(ax, geometry)
        draw_loads(ax, live_loads, geometry["L_total"])
        n = len(live_loads)
        ax.set_title(f"Beam with {n} load{'s' if n != 1 else ''} applied",
                     fontsize=11, pad=8)
        plt.tight_layout()
        plt.show()

# ── Button callbacks ──────────────────────────────────────────────────────
def on_add_clicked(b):
    L = geometry["L_total"]
    with out_status:
        out_status.clear_output(wait=True)
        if dd_type.value == "point":
            x   = sl_x.value
            mag = sl_P.value
            if x < 0 or x > L:
                print(f"Error: x={x} is outside the beam.")
                return
            live_loads.append({"type": "point", "x": x, "magnitude": mag})
            print(f"✓ Added point load {mag:+.0f} kN at x = {x:.1f} m")
        else:
            x1, x2, w = sl_x1.value, sl_x2.value, sl_w.value
            if x2 <= x1:
                print("Error: UDL end must be greater than UDL start.")
                return
            live_loads.append({"type": "udl", "x1": x1, "x2": x2, "w": w})
            print(f"✓ Added UDL {w:+.0f} kN/m from {x1:.1f} m to {x2:.1f} m")
    redraw()

def on_clear_clicked(b):
    live_loads.clear()
    with out_status:
        out_status.clear_output(wait=True)
        print("All loads cleared.")
    redraw()

btn_add.on_click(on_add_clicked)
btn_clear.on_click(on_clear_clicked)

# ── Initial render + display ──────────────────────────────────────────────
redraw()
display(widgets.VBox([
    widgets.HTML("<b>Define loads and click 'Add Load':</b>"),
    dd_type,
    pt_controls,
    udl_controls,
    widgets.HBox([btn_add, btn_clear]),
    out_status,
    out_plot
]))

**Try this:**
1. Add a **−20 kN point load** at x = 5 m (midspan)
2. Switch to **UDL**, set it from 2 m to 8 m at −5 kN/m, add it
3. Add a **−8 kN point load** at x = 1 m (on the left cantilever)
4. Click **Clear All** and start fresh

Notice that negative loads point **downward** and positive loads point **upward**.

---

## 3.5 Practice Exercise

Without the widget, write code that builds a load list for this scenario:
- A **−30 kN point load** at the midpoint of the interior span (x = 5 m)
- A **−4 kN/m UDL** covering the entire beam (0 to 10 m)
- A **−15 kN point load** at the tip of the right cantilever (x = 9 m)

Then loop over the list and print a summary of all loads.

In [ ]:
# Your code here


---

## Summary

**Python concepts covered:**
- **Dictionaries** `{}` — key–value pairs for grouping related data
- **Lists of dictionaries** — storing multiple loads in one collection
- **Functions** `def` — reusable named code blocks with parameters and return values
- **`for` loop over a list** — processing every load
- **`if/elif`** — branching on load type
- **Button widgets** — triggering actions on click with `.on_click()`
- **`widgets.Output()`** — capturing output inside button callbacks

**Structural concepts covered:**
- Sign convention: downward loads = negative, upward reactions = positive
- Point loads vs UDL — data model and drawing convention
- Validation: loads must lie within the beam span

## Before the Next Lesson

In **Lesson 4** you'll solve for the support reactions using two equilibrium equations. Think about: if you place a load on the left cantilever, which reaction will be larger — A or B?